In [ ]:
import inspect
import os
import sys
from collections import Counter
from pathlib import Path

import analysis
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rich.pretty import pprint

# In Jupyter, __file__ is not defined, so use the current working directory
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import matcher

from frankenstein.tools import arithmetic, data_retrieval

ARITHMETIC_TOOL_NAMES = [name for name, _ in inspect.getmembers(arithmetic, predicate=inspect.isfunction)]
DATA_TOOL_NAMES = [name for name, _ in inspect.getmembers(data_retrieval, predicate=inspect.isfunction)]

run_dir = Path('runs')
dfs = {
    f.stem: pd.read_json(f, orient='records', lines=True, precise_float=True)
    for f in run_dir.iterdir()
    if f.is_file() and 'answerable-full_all-tools_0-shot' in f.name
}
print(f'Found {len(dfs)} runs in {run_dir}:')

m = matcher.Matcher()

In [ ]:
from tqdm import tqdm

summary_rows = []
for run_name, run_df in tqdm(dfs.items(), desc='Processing runs', unit='run'):
    # Compute tool call columns if not already present
    # Keep only rows where df['answer_format'] is not bool
    # run_df = run_df[run_df['answer_format'] != bool]
    if 'gold_tool_calls' not in run_df.columns:
        tools = DATA_TOOL_NAMES if 'data-tools' in run_name else []
        run_df['gold_tool_calls'] = run_df.apply(lambda row: analysis.get_gold_tool_calls(row, tools), axis=1)
    if 'pred_tool_calls' not in run_df.columns:
        run_df['pred_tool_calls'] = run_df.apply(analysis.get_pred_tool_calls, axis=1)
    if 'true_positives' not in run_df.columns:
        run_df['true_positives'] = run_df.apply(
            analysis.get_true_positives, axis=1
        )  # True positives are predicted tool calls that were in the gold calls
    if 'false_positives' not in run_df.columns:
        run_df['false_positives'] = run_df.apply(
            analysis.get_false_positives, axis=1
        )  # False positives are predicted tool calls that were not in the gold calls
    if 'false_negatives' not in run_df.columns:
        run_df['false_negatives'] = run_df.apply(
            analysis.get_false_negatives, axis=1
        )  # False negatives are gold tool calls that were not predicted
    if 'precision' not in run_df.columns:
        run_df['precision'] = run_df.apply(analysis.get_precision, axis=1)
    if 'coverage' not in run_df.columns:
        run_df['coverage'] = run_df.apply(analysis.get_coverage, axis=1)
    # if 'error_made' not in run_df.columns:
    #     run_df['error_made'] = run_df.apply(analysis.get_error_made, axis=1)
    # if 'correct_indicator_data_process' not in run_df.columns:
    #     run_df['correct_indicator_data_process'] = run_df.apply(analysis.get_correct_indicator_data_process, axis=1)

    summary = {
        'run': run_name,
        'n': len(run_df),
        'accuracy': run_df['correct'].mean() if 'correct' in run_df.columns else None,
        'precision_mean': run_df['precision'].mean(),
        'precision_std': run_df['precision'].std(),
        'coverage_mean': run_df['coverage'].mean(),
        'coverage_std': run_df['coverage'].std(),
        # 'error_rate': run_df['error_made'].mean(),
        # 'correct_indicator_data_process': run_df['correct_indicator_data_process'].mean(),
    }
    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(by='run')
summary_df = summary_df.round(2)  # Round to 2 decimal places

# Drop certain rows from the summary depending on the run name
summary_df = summary_df[~summary_df['run'].str.contains('Llama-3.1-70B|Llama-3.1-8B|Llama-3.2-3B')]
summary_df['precision'] = summary_df.apply(lambda row: rf'{row["precision_mean"]}$\pm${row["precision_std"]}', axis=1)
summary_df['coverage'] = summary_df.apply(lambda row: rf'{row["coverage_mean"]}$\pm${row["coverage_std"]}', axis=1)
summary_df = summary_df.drop(columns=['precision_mean', 'precision_std', 'coverage_mean', 'coverage_std'])
normal_runs = summary_df[~summary_df['run'].str.contains('data-tools|partial')]
zero_shot = normal_runs[normal_runs['run'].str.contains('0-shot')]
one_shot = normal_runs[normal_runs['run'].str.contains('1-shot')]
three_shot = normal_runs[normal_runs['run'].str.contains('3-shot')]
partial_runs = summary_df[summary_df['run'].str.contains('partial')]
data_tool_runs = summary_df[summary_df['run'].str.contains('data-tools')]
normal_runs

In [ ]:
df = dfs['Qwen3-4B_answerable-full_all-tools_0-shot']
df.head()

In [ ]:
df = dfs['Qwen3-4B_answerable-full_all-tools_0-shot']
i = 40
print('Gold tool calls:', flush=True)
pprint(Counter([c['name'] for c in df.iloc[i]['gold_tool_calls']]))
# pprint(df.iloc[i]['gold_tool_calls'])
print('Pred tool calls:', flush=True)
pprint(Counter([c['name'] for c in df.iloc[i]['pred_tool_calls']]))
# pprint(df.iloc[i]['pred_tool_calls'])
print('False positive factor:', flush=True)
pprint(analysis.false_positive_overcall_factor(df.iloc[i]))
print('False negative factor:', flush=True)
pprint(analysis.false_negative_undercall_factor(df.iloc[i]))

In [ ]:
ALL_TOOL_NAMES = [name for name in DATA_TOOL_NAMES if name not in ('get_indicator_name_from_code')] + [
    name for name in ARITHMETIC_TOOL_NAMES if name not in ('multiply', 'less_than', 'index', 'rank')
]

In [ ]:
run_df.sample(1)['question'].values[0]

In [ ]:
# placeholders to store factors for each tool across different model runs
positive_factors = []
# positive_factors_deduped = []
negative_factors = []

for run_name, run_df in dfs.items():
    run_df['false_positive_factors'] = run_df.apply(
        lambda row: analysis.false_positive_overcall_factor(
            row,
        ),
        axis=1,
    )
    # run_df['false_positive_factors_deduped'] = run_df.apply(
    #     lambda row: analysis.false_positive_overcall_factor_signature(row, remove_duplicates=True, include_equal=True),
    #     axis=1,
    # )
    run_df['false_negative_factors'] = run_df.apply(analysis.false_negative_undercall_factor, axis=1)

    fpf = analysis.aggregate_tool_factors(run_df['false_positive_factors'])
    positive_factors.append({'run_name': run_name.replace('_answerable-full_all-tools_0-shot', ''), **fpf})

    # fpf_deduped = analysis.aggregate_tool_factors(run_df['false_positive_factors_deduped'])
    # positive_factors_deduped.append({'run_name': run_name.replace('_answerable-full_all-tools_0-shot', ''), **fpf_deduped})

    fnf = analysis.aggregate_tool_factors(run_df['false_negative_factors'])
    negative_factors.append({'run_name': run_name.replace('_answerable-full_all-tools_0-shot', ''), **fnf})

positive_factors_df = pd.DataFrame(positive_factors).set_index('run_name')
# positive_factors_deduped_df = pd.DataFrame(positive_factors_deduped).set_index('run_name')
negative_factors_df = pd.DataFrame(negative_factors).set_index('run_name')

# reindex and fill missing tools with 0
positive_factors_df = positive_factors_df.reindex(columns=ALL_TOOL_NAMES, fill_value=0)
# positive_factors_deduped_df = positive_factors_deduped_df.reindex(columns=ALL_TOOL_NAMES, fill_value=0)
negative_factors_df = negative_factors_df.reindex(columns=ALL_TOOL_NAMES, fill_value=0)
positive_factors_df = positive_factors_df.fillna(1)
# positive_factors_deduped_df = positive_factors_deduped_df.fillna(1)
negative_factors_df = negative_factors_df.fillna(1)
positive_factors_df = positive_factors_df.round(2)
# positive_factors_deduped_df = positive_factors_deduped_df.round(2)
negative_factors_df = negative_factors_df.round(2)


run_rename_mapping = {
    'Qwen3-4B': 'Qwen 3 4B',
    'Qwen3-14B': 'Qwen 3 14B',
    'Llama-3.1-8B-Instruct': 'Llama 3.1 8B Instruct',
    'Llama-3.3-70B-Instruct': 'Llama 3.3 70B Instruct',
    'Mistral-Small-3.1-24B': 'Mistral Small 3.1 24B',
    'Qwen3-30B-A3B': 'Qwen 3 30B A3B',
    'Qwen3-32B': 'Qwen 3 32B',
    'gpt-4.1-mini': 'GPT 4.1 Mini',
    'gpt-4o-mini': 'GPT 4o Mini',
    'gpt-5-mini': 'GPT 5 Mini',
    'gpt-5-nano': 'GPT 5 Nano',
    'Llama-3.1-70B-Instruct': 'Llama 3.1 70B Instruct',
    'Llama-3.2-3B-Instruct': 'Llama 3.2 3B Instruct',
}

tool_col_rename_mapping = {
    'greater_than': 'comparison',
}
positive_factors_df = positive_factors_df.rename(columns=tool_col_rename_mapping)
# positive_factors_deduped_df = positive_factors_deduped_df.rename(columns=tool_col_rename_mapping)
negative_factors_df = negative_factors_df.rename(columns=tool_col_rename_mapping)
positive_factors_df.index = positive_factors_df.index.to_series().replace(run_rename_mapping)
# positive_factors_deduped_df.index = positive_factors_deduped_df.index.to_series().replace(run_rename_mapping)
negative_factors_df.index = negative_factors_df.index.to_series().replace(run_rename_mapping)
# Set custom order for the index based on the order in run_rename_mapping values
# Order =
# GPT 4.1 Mini, GPT 4o Mini, GPT 5 Mini, GPT 5 Nano, Qwen 3 4B, Qwen 3 14B, Qwen 3 30B A3B, Qwen 3 32B, Mistral Small 3.1 24B, Llama 3.1 8B Instruct, Llama 3.1 70B Instruct, Llama 3.2 3B Instruct, Llama 3.3 70B Instruct

custom_order = [
    'GPT 4o Mini',
    'GPT 4.1 Mini',
    'GPT 5 Mini',
    'GPT 5 Nano',
    'Llama 3.1 8B Instruct',
    'Llama 3.1 70B Instruct',
    'Llama 3.2 3B Instruct',
    'Llama 3.3 70B Instruct',
    'Mistral Small 3.1 24B',
    'Qwen 3 4B',
    'Qwen 3 14B',
    'Qwen 3 30B A3B',
    'Qwen 3 32B',
]
positive_factors_df = positive_factors_df.reindex(custom_order)
# positive_factors_deduped_df = positive_factors_deduped_df.reindex(custom_order)
negative_factors_df = negative_factors_df.reindex(custom_order)

In [ ]:
DF = negative_factors_df
bound = 0.5
# cols = [col for col in DF.columns if col in ARITHMETIC_TOOL_NAMES]
cols = DF.columns
# count number of cells in the dataframe greater than or equal to x
count_ge_x = (bound <= DF[cols]).sum().sum()
count_le_x = (bound >= DF[cols]).sum().sum()
total_cells = DF[cols].size
print(f'Number of cells in the dataframe greater than or equal to {bound}: {count_ge_x}')
print(f'Number of cells in the dataframe less than or equal to {bound}: {count_le_x}')
print(f'Total number of cells in the dataframe: {total_cells}')
print(f'Percentage of cells greater than or equal to {bound}: {(count_ge_x / total_cells * 100):.2f}%')
print(f'Percentage of cells less than or equal to {bound}: {(count_le_x / total_cells * 100):.2f}%')

In [ ]:
def plot_factor_heatmap(
    DF,
    save_name,
    title,
    colorbar_label,
    mode='positive',
    cmap=None,
    figsize=(12, 4),
    fmt='{:.2f}',
    fontfamily='monospace',
    gap_fraction=0.5,  # increase this to make the gap look wider
):
    if DF is None or DF.size == 0:
        raise ValueError('DF is empty or None')

    if cmap is None:
        cmap = 'viridis' if mode == 'positive' else 'viridis_r'

    # vmin/vmax from original matrix only
    abs_max = np.nanmax(np.abs(DF.values))
    abs_min = np.nanmin(np.abs(DF.values))

    if mode == 'positive':
        vmin, vmax = 1, abs_max
    else:
        vmin, vmax = abs_min, 1

    models = DF.index.tolist()
    tools = DF.columns.tolist()
    n_models = len(models)
    n_tools = len(tools)

    row_avgs = DF.mean(axis=1)
    col_avgs = DF.mean(axis=0)

    # Build expanded matrix: models+Avg row, tools+Avg col
    # Start with NaNs so bottom-right stays NaN (blank)
    data = np.full((n_models + 1, n_tools + 1), np.nan)
    data[0:n_models, 0:n_tools] = DF.values  # main block
    data[0:n_models, -1] = row_avgs.values  # avg column
    data[-1, 0:n_tools] = col_avgs.values  # avg row
    # data[-1, -1] remains NaN → blank overall-average cell

    # Colormap that treats NaNs (bottom-right + any others) as white/blank
    base_cmap = plt.cm.get_cmap(cmap)
    cmap_with_nan = base_cmap.copy()
    cmap_with_nan.set_bad(color='white')

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(data, cmap=cmap_with_nan, vmin=vmin, vmax=vmax, aspect='auto')

    # Remove outer border/spines
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Ticks / labels
    ax.set_xticks(list(range(n_tools)) + [n_tools])
    ax.set_xticklabels(
        tools + ['Avg'],
        rotation=45,
        ha='right',
        fontfamily=fontfamily,
    )

    ax.set_yticks(list(range(n_models)) + [n_models])
    ax.set_yticklabels(
        models + ['Avg'],
        # fontfamily=fontfamily,
    )

    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=colorbar_label)

    # Separator positions between main block and averages
    sep_x = n_tools - 0.5
    sep_y = n_models - 0.5

    # Draw separator lines only over the main block edge
    # so they don't run into the bottom-right "overall avg" cell
    ax.vlines(
        sep_x,
        ymin=-0.5,
        ymax=n_models - 0.5,
        color='white',
        linewidth=gap_fraction * 6,
    )
    ax.hlines(
        sep_y,
        xmin=-0.5,
        xmax=n_tools - 0.5,
        color='white',
        linewidth=gap_fraction * 6,
    )

    # Annotation helper
    def annotate_cell(i, j, value):
        if np.isnan(value):
            return
        try:
            text_val = fmt.format(value)
        except Exception:
            text_val = f'{value:.2f}'
        text = ax.text(j, i, text_val, ha='center', va='center', color='white')
        text.set_path_effects([path_effects.withStroke(linewidth=1, foreground='black')])

    # Annotate everything except the bottom-right NaN cell
    for i in range(n_models + 1):
        for j in range(n_tools + 1):
            annotate_cell(i, j, data[i, j])

    plt.savefig(save_name, bbox_inches='tight', dpi=300)
    plt.show()
    plt.close(fig)

In [ ]:
# def plot_factor_heatmap(
#     DF,
#     save_name,
#     title,
#     colorbar_label,
#     mode='positive',
#     cmap=None,
#     figsize=(12, 4),
#     fmt='{:.2f}',
#     fontfamily='monospace',
# ):
#     """DF: pandas DataFrame (models x tools)
#     mode: 'positive' -> vmin=1, vmax=max_abs; 'negative' -> vmin=min_abs, vmax=1
#     other args: as expected
#     """
#     if cmap is None:
#         cmap = 'viridis' if mode == 'positive' else 'viridis_r'

#     # compute vmin/vmax defaults
#     abs_max = np.nanmax(np.abs(DF.values)) if DF.size else 1
#     abs_min = np.nanmin(np.abs(DF.values)) if DF.size else 1
#     if mode == 'positive':
#         vmin, vmax = 1, abs_max
#     else:
#         vmin, vmax = abs_min, 1

#     datasets = DF.index.tolist()
#     tools = DF.columns.tolist()
#     VALUES = DF

#     fig, ax = plt.subplots(figsize=figsize)
#     im = ax.imshow(VALUES.values, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')
#     ax.set_xticks(np.arange(len(tools)))
#     ax.set_yticks(np.arange(len(datasets)))
#     ax.set_xticklabels(tools, rotation=45, ha='right', fontfamily=fontfamily)
#     ax.set_yticklabels(datasets)
#     ax.set_title(title)
#     fig.colorbar(im, ax=ax, label=colorbar_label)

#     # annotations
#     for i in range(len(datasets)):
#         for j in range(len(tools)):
#             try:
#                 text_val = fmt.format(VALUES.iloc[i, j])
#             except Exception:
#                 text_val = f'{VALUES.iloc[i, j]:.2f}'
#             text = ax.text(j, i, text_val, ha='center', va='center', color='white')
#             text.set_path_effects([path_effects.withStroke(linewidth=1, foreground='black')])

#     plt.savefig(save_name, bbox_inches='tight', dpi=300)
#     plt.show()
#     plt.close(fig)


# pre-built arg dicts for easy swapping
positive_factors_df_args = {
    'DF': positive_factors_df,
    'save_name': 'false_positive_overcall_factor_heatmap.pdf',
    'title': 'False Positive Overcall Factor by Tool and Model',
    'colorbar_label': 'False Positive Overcall Factor',
    'mode': 'positive',
    'cmap': 'viridis',
    'figsize': (12, 4),
    'fmt': '{:.2f}',
}

# sig_positive_factors_df_dedup_args = {
#     'DF': positive_factors_deduped_df,
#     'save_name': 'false_positive_overcall_factor_signature_dedup_heatmap.pdf',
#     'title': 'Signature-dedup False Positive Overcall Factor by Tool and Model',
#     'colorbar_label': 'False Positive Overcall Factor (signature-dedup)',
#     'mode': 'positive',
#     'cmap': 'viridis',
#     'figsize': (12, 4),
#     'fmt': '{:.2f}',
# }

negative_factors_df_args = {
    'DF': negative_factors_df,
    'save_name': 'false_negative_undercall_factor_heatmap.pdf',
    'title': 'False Negative Undercall Factor by Tool and Model',
    'colorbar_label': 'False Negative Undercall Factor',
    'mode': 'negative',
    'cmap': 'viridis_r',
    'figsize': (12, 4),
    'fmt': '{:.2f}',
}

# Example usage: choose which dict to pass
plot_factor_heatmap(**positive_factors_df_args)
# plot_factor_heatmap(**sig_positive_factors_df_dedup_args)
plot_factor_heatmap(**negative_factors_df_args)
